# Optics — From Geometric Rays to Wave Phenomena

Light is an **electromagnetic wave** whose electric and magnetic fields oscillate perpendicular to its direction of propagation. The fundamental relation linking wavelength $\lambda$, frequency $\nu$, and the speed of light $c$ in vacuum is:

$$c = \lambda \, \nu \approx 3 \times 10^8 \;\text{m/s}$$

In a medium with refractive index $n$, light travels at reduced speed $v = c/n$, while the frequency remains constant and the wavelength shortens to $\lambda_{\text{med}} = \lambda_0 / n$.

**Two complementary descriptions:**

| Regime | Model | Valid when |
|:---|:---|:---|
| **Geometric (ray) optics** | Light as straight rays obeying reflection/refraction laws | Objects $\gg \lambda$ — lenses, mirrors, prisms |
| **Wave (physical) optics** | Light as EM wave exhibiting interference, diffraction, polarization | Feature sizes $\sim \lambda$ — slits, thin films, gratings |

The visible spectrum spans roughly $380\;\text{nm}$ (violet) to $750\;\text{nm}$ (red). This notebook progresses from geometric optics (refraction, lenses, mirrors) through wave optics (interference, diffraction, polarization) with interactive visualizations at every stage.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.patches as patches
from matplotlib.lines import Line2D

%matplotlib inline
plt.rcParams.update({
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
    'figure.dpi': 100, 'font.size': 10,
    'axes.labelsize': 10, 'axes.titlesize': 11,
})

## Refraction & Snell's Law

When light crosses the boundary between two media with different refractive indices, it changes direction. **Snell's law** relates the angles of incidence and refraction:

$$n_1 \sin\theta_1 = n_2 \sin\theta_2$$

where $\theta_1$ is the angle of incidence (measured from the normal) and $\theta_2$ is the angle of refraction.

**Key consequences:**

- Light bends **toward** the normal when entering a denser medium ($n_2 > n_1$)
- Light bends **away** from the normal when entering a less dense medium ($n_2 < n_1$)
- **Critical angle:** when $n_1 > n_2$, there exists a critical angle $\theta_c = \arcsin(n_2 / n_1)$ beyond which all light is reflected — **total internal reflection (TIR)**

| Medium | Refractive index $n$ |
|:---|:---|
| Vacuum | 1.000 |
| Air | 1.0003 |
| Water | 1.333 |
| Glass (crown) | 1.52 |
| Diamond | 2.42 |

> Adjust $\theta_1$, $n_1$, and $n_2$ below to see refraction, and watch for total internal reflection when $\theta_1 > \theta_c$.

In [2]:
def build_snell_demo():
    style = {'description_width': 'initial'}
    sl = dict(style=style, layout=widgets.Layout(width='400px'))
    theta_w = widgets.FloatSlider(value=30, min=0, max=89, step=1,
                                  description='Angle of incidence (deg)', **sl)
    n1_w = widgets.FloatSlider(value=1.5, min=1.0, max=2.5, step=0.01,
                               description='n1 (top medium)', **sl)
    n2_w = widgets.FloatSlider(value=1.0, min=1.0, max=2.5, step=0.01,
                               description='n2 (bottom medium)', **sl)
    out = widgets.Output()

    def update(_):
        theta1_deg = theta_w.value
        n1, n2 = n1_w.value, n2_w.value
        theta1 = np.radians(theta1_deg)

        sin_theta2 = n1 * np.sin(theta1) / n2
        tir = sin_theta2 > 1.0
        has_critical = n1 > n2
        theta_c_deg = np.degrees(np.arcsin(n2 / n1)) if has_critical else None

        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(1, 1, figsize=(8, 6))
            ax.set_xlim(-3, 3)
            ax.set_ylim(-3, 3)
            ax.set_aspect('equal')
            ax.grid(False)

            # Draw interface and media
            ax.axhline(0, color='black', lw=2)
            ax.fill_between([-3, 3], 0, 3, alpha=0.08, color='steelblue')
            ax.fill_between([-3, 3], -3, 0, alpha=0.08, color='darkorange')
            ax.text(-2.8, 2.6, f'n1 = {n1:.2f}', fontsize=11, color='steelblue', fontweight='bold')
            ax.text(-2.8, -2.8, f'n2 = {n2:.2f}', fontsize=11, color='darkorange', fontweight='bold')

            # Normal (dashed)
            ax.plot([0, 0], [-2.5, 2.5], 'k--', lw=1, alpha=0.5)
            ax.text(0.1, 2.3, 'normal', fontsize=8, alpha=0.6)

            # Incident ray
            ray_len = 2.5
            ix = -ray_len * np.sin(theta1)
            iy = ray_len * np.cos(theta1)
            ax.annotate('', xy=(0, 0), xytext=(ix, iy),
                        arrowprops=dict(arrowstyle='->', color='steelblue', lw=2.5))
            ax.text(ix * 0.5 - 0.3, iy * 0.5 + 0.15, 'incident', fontsize=9, color='steelblue')

            # Angle arc for theta1
            arc1 = patches.Arc((0, 0), 1.2, 1.2, angle=0, theta1=90 - theta1_deg, theta2=90,
                               color='steelblue', lw=1.5)
            ax.add_patch(arc1)
            ax.text(0.15, 0.85, f'{theta1_deg:.0f}°', fontsize=9, color='steelblue')

            # Reflected ray
            rx = ray_len * np.sin(theta1)
            ry = ray_len * np.cos(theta1)
            alpha_r = 1.0 if tir else 0.35
            ax.annotate('', xy=(rx, ry), xytext=(0, 0),
                        arrowprops=dict(arrowstyle='->', color='gray', lw=2 if tir else 1.5, alpha=alpha_r))
            if tir:
                ax.text(rx * 0.5 + 0.15, ry * 0.5 + 0.1, 'TIR', fontsize=10,
                        color='#d62728', fontweight='bold')

            if not tir:
                theta2 = np.arcsin(sin_theta2)
                theta2_deg = np.degrees(theta2)
                tx = ray_len * np.sin(theta2)
                ty = -ray_len * np.cos(theta2)
                ax.annotate('', xy=(tx, ty), xytext=(0, 0),
                            arrowprops=dict(arrowstyle='->', color='darkorange', lw=2.5))
                ax.text(tx * 0.5 + 0.15, ty * 0.5 - 0.15, 'refracted', fontsize=9, color='darkorange')

                arc2 = patches.Arc((0, 0), 1.6, 1.6, angle=0, theta1=270, theta2=270 + theta2_deg,
                                   color='darkorange', lw=1.5)
                ax.add_patch(arc2)
                ax.text(0.15, -1.1, f'{theta2_deg:.1f}°', fontsize=9, color='darkorange')

                title = f"Snell's Law: {n1:.2f} sin({theta1_deg:.0f}°) = {n2:.2f} sin({theta2_deg:.1f}°)"
            else:
                title = f"Total Internal Reflection — angle ({theta1_deg:.0f}°) exceeds critical angle ({theta_c_deg:.1f}°)"

            if has_critical:
                title += f'  |  Critical angle = {theta_c_deg:.1f}°'

            ax.set_title(title, fontsize=10, color='#d62728' if tir else 'black')
            ax.set_xlabel('x'); ax.set_ylabel('y')
            plt.tight_layout()
            plt.show()

    for w in (theta_w, n1_w, n2_w):
        w.observe(update, names='value')
    display(widgets.VBox([theta_w, n1_w, n2_w, out]))
    update(None)

build_snell_demo()

## Dispersion & Prisms

The refractive index of a material is not constant — it depends on wavelength. This phenomenon is called **dispersion** and is responsible for prisms splitting white light into a rainbow.

The **Cauchy equation** approximates $n(\lambda)$ for transparent materials:

$$n(\lambda) = A + \frac{B}{\lambda^2} + \frac{C}{\lambda^4} + \cdots$$

where $A$, $B$, $C$ are material-dependent constants and $\lambda$ is in micrometers. Since shorter wavelengths (blue/violet) have higher $n$, they are refracted more strongly than longer wavelengths (red).

**Prism dispersion:** a triangular prism with apex angle $\alpha$ deviates a ray by the **deviation angle** $\delta$. At minimum deviation:

$$n = \frac{\sin\!\left(\frac{\alpha + \delta_{\min}}{2}\right)}{\sin\!\left(\frac{\alpha}{2}\right)}$$

Each wavelength experiences a different $n$, hence a different $\delta$, producing the familiar spectral spread.

> Adjust the apex angle below to see how the prism separates white light into its constituent colors.

In [ ]:
def build_prism_demo():
    style = {'description_width': 'initial'}
    sl = dict(style=style, layout=widgets.Layout(width='420px'))
    apex_w = widgets.FloatSlider(value=60, min=20, max=80, step=1,
                                 description='Apex angle (deg)', **sl)
    out = widgets.Output()

    def wavelength_to_rgb(wl):
        """Convert wavelength (nm) to approximate RGB."""
        if 380 <= wl < 440:
            r, g, b = -(wl - 440) / (440 - 380), 0.0, 1.0
        elif 440 <= wl < 490:
            r, g, b = 0.0, (wl - 440) / (490 - 440), 1.0
        elif 490 <= wl < 510:
            r, g, b = 0.0, 1.0, -(wl - 510) / (510 - 490)
        elif 510 <= wl < 580:
            r, g, b = (wl - 510) / (580 - 510), 1.0, 0.0
        elif 580 <= wl < 645:
            r, g, b = 1.0, -(wl - 645) / (645 - 580), 0.0
        elif 645 <= wl <= 780:
            r, g, b = 1.0, 0.0, 0.0
        else:
            r, g, b = 0.0, 0.0, 0.0
        # Intensity falloff at edges
        if 380 <= wl < 420:
            factor = 0.3 + 0.7 * (wl - 380) / (420 - 380)
        elif 645 < wl <= 780:
            factor = 0.3 + 0.7 * (780 - wl) / (780 - 645)
        else:
            factor = 1.0
        return (r * factor, g * factor, b * factor)

    def cauchy_n(wl_nm, A=1.522, B=4200):
        """Cauchy equation: n = A + B/lambda^2, lambda in nm."""
        return A + B / (wl_nm ** 2)

    def update(_):
        alpha_deg = apex_w.value
        alpha = np.radians(alpha_deg)

        with out:
            clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

            # Left panel: n(lambda) curve
            wls = np.linspace(380, 750, 300)
            ns = cauchy_n(wls)
            ax1.plot(wls, ns, 'k-', lw=2)
            for wl in [400, 450, 500, 550, 600, 650, 700]:
                c = wavelength_to_rgb(wl)
                ax1.plot(wl, cauchy_n(wl), 'o', color=c, ms=10, zorder=5)
            ax1.set_xlabel('Wavelength (nm)')
            ax1.set_ylabel('Refractive index n')
            ax1.set_title('Cauchy Dispersion: n(λ) = A + B/λ²')
            ax1.set_xlim(370, 760)

            # Right panel: prism ray trace
            ax2.set_xlim(-1, 5)
            ax2.set_ylim(-2, 2.5)
            ax2.set_aspect('equal')
            ax2.grid(False)

            # Draw prism (equilateral-ish triangle with given apex)
            h = 2.0
            half_base = h * np.tan(alpha / 2)
            prism_x = [0, h, 0, 0]
            prism_y = [-half_base, 0, half_base, -half_base]
            ax2.fill(prism_x, prism_y, alpha=0.15, color='steelblue')
            ax2.plot(prism_x, prism_y, 'steelblue', lw=2)

            # Trace rays for several wavelengths
            wavelengths = np.linspace(400, 700, 8)
            inc_angle = np.radians(45)

            # Left face normal points to (-1, 0) direction rotated
            face_angle = np.arctan2(half_base, h)  # angle of left face from vertical
            normal_angle = face_angle  # normal to left face, pointing left

            for wl in wavelengths:
                n = cauchy_n(wl)
                color = wavelength_to_rgb(wl)

                # Angle of incidence on first face (measured from face normal)
                theta_i1 = inc_angle - face_angle
                if theta_i1 < 0:
                    continue
                sin_t = np.sin(theta_i1) / n
                if abs(sin_t) > 1:
                    continue
                theta_r1 = np.arcsin(sin_t)

                # Ray enters prism — find intersection with second face
                # Second face: from (0, half_base) to (h, 0)
                # Angle inside prism relative to horizontal
                ray_angle_inside = face_angle - theta_r1

                # Hit point on left face
                hit_y = half_base * 0.3  # approximate entry point
                hit_x = (half_base - hit_y) / half_base * h if half_base > 0 else 0

                # Entry point on left face
                t_entry = 0.3  # parameter along face
                entry_x = 0
                entry_y = half_base - 2 * half_base * t_entry

                # Direction inside prism
                dir_x = np.cos(ray_angle_inside)
                dir_y = -np.sin(ray_angle_inside)

                # Find exit on right face: from (h, 0) to (0, -half_base)
                # Right face: parametric: (h - h*s, -half_base*s) for s in [0,1]
                # But simpler: the right face has the same angle, mirrored
                # Angle of incidence on second face
                theta_i2 = alpha - theta_r1
                if theta_i2 < 0:
                    continue
                sin_t2 = n * np.sin(theta_i2)
                if abs(sin_t2) > 1:
                    continue  # TIR inside prism
                theta_r2 = np.arcsin(sin_t2)

                # Deviation angle
                delta = theta_i1 + theta_r2 - alpha

                # Draw simplified ray diagram
                # Incoming ray
                start_x = -0.8
                start_y = entry_y + 0.8 * np.tan(inc_angle - face_angle + face_angle)
                ax2.plot([start_x, entry_x], [start_y, entry_y], color=color, lw=2, alpha=0.9)

                # Ray inside prism (approximate)
                exit_frac = 0.6
                exit_x = h * (1 - exit_frac * 0.3)
                exit_y = -half_base * exit_frac * 0.3
                # Better: just compute internal path
                internal_len = 1.5
                mid_x = entry_x + internal_len * dir_x
                mid_y = entry_y + internal_len * dir_y
                ax2.plot([entry_x, mid_x], [entry_y, mid_y], color=color, lw=1.8, alpha=0.7)

                # Exiting ray
                exit_dir_angle = -face_angle + theta_r2
                end_x = mid_x + 2.5 * np.cos(exit_dir_angle)
                end_y = mid_y - 2.5 * np.sin(exit_dir_angle)
                ax2.plot([mid_x, end_x], [mid_y, end_y], color=color, lw=2, alpha=0.9)

            ax2.set_title(f'Prism Dispersion (apex = {alpha_deg:.0f}°)', fontsize=11)
            ax2.text(0.3, 0, 'PRISM', fontsize=10, ha='center', color='steelblue', fontweight='bold')

            # Legend
            for wl, label in [(400, '400 nm'), (550, '550 nm'), (700, '700 nm')]:
                ax2.plot([], [], color=wavelength_to_rgb(wl), lw=2, label=label)
            ax2.legend(fontsize=9, loc='upper right')

            plt.tight_layout()
            plt.show()

    apex_w.observe(update, names='value')
    display(widgets.VBox([apex_w, out]))
    update(None)

build_prism_demo()

## Thin Lens Equation

A **thin lens** refracts light so that all rays from a point object converge to (or appear to diverge from) a single image point. The relationship between object distance $d_o$, image distance $d_i$, and focal length $f$ is:

$$\frac{1}{f} = \frac{1}{d_o} + \frac{1}{d_i}$$

The **transverse magnification** is:

$$M = -\frac{d_i}{d_o}$$

**Sign conventions** (real-is-positive):

| Quantity | Positive | Negative |
|:---|:---|:---|
| $f$ | Converging (convex) lens | Diverging (concave) lens |
| $d_o$ | Object on incoming side (real) | Virtual object |
| $d_i$ | Image on outgoing side (real) | Image on incoming side (virtual) |
| $M$ | Upright image | Inverted image |

**Three principal rays** for a converging lens:
1. Parallel to axis $\rightarrow$ passes through back focal point
2. Through the center $\rightarrow$ continues undeviated
3. Through front focal point $\rightarrow$ exits parallel to axis

> Move the object distance and focal length sliders to see how the image forms. Watch for virtual images when $d_o < f$ for a converging lens.

In [ ]:
def build_lens_demo():
    style = {'description_width': 'initial'}
    sl = dict(style=style, layout=widgets.Layout(width='420px'))
    do_w = widgets.FloatSlider(value=6, min=0.5, max=15, step=0.1,
                               description='Object distance do', **sl)
    f_w = widgets.FloatSlider(value=3, min=-6, max=8, step=0.1,
                              description='Focal length f', **sl)
    out = widgets.Output()

    def update(_):
        do = do_w.value
        f = f_w.value
        if abs(do - f) < 0.01:
            do += 0.02  # avoid infinity

        di = 1.0 / (1.0 / f - 1.0 / do) if abs(1.0 / f - 1.0 / do) > 1e-6 else 1e6
        M = -di / do
        real_image = di > 0
        obj_h = 1.5  # object height
        img_h = M * obj_h

        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(1, 1, figsize=(14, 5))

            # Axis range
            x_min = min(-do - 2, -2)
            x_max = max(di + 2, 2, 8)
            if abs(di) > 30:
                x_max = 15
                x_min = -do - 2
            ax.set_xlim(x_min, x_max)
            y_lim = max(abs(obj_h), abs(img_h) if abs(img_h) < 10 else obj_h, 3) + 1
            ax.set_ylim(-y_lim, y_lim)
            ax.set_aspect('equal')
            ax.grid(False)

            # Optical axis
            ax.axhline(0, color='black', lw=0.8, alpha=0.5)

            # Lens
            lens_h = y_lim * 0.85
            if f > 0:
                ax.annotate('', xy=(0, lens_h), xytext=(0, -lens_h),
                            arrowprops=dict(arrowstyle='<->', color='steelblue', lw=2.5))
            else:
                # Diverging lens — draw with inward arrows
                ax.plot([0, 0], [-lens_h, lens_h], color='steelblue', lw=2.5)
                ax.plot([-0.2, 0, 0.2], [lens_h - 0.3, lens_h, lens_h - 0.3], 'steelblue', lw=2)
                ax.plot([-0.2, 0, 0.2], [-lens_h + 0.3, -lens_h, -lens_h + 0.3], 'steelblue', lw=2)

            # Focal points
            ax.plot(f, 0, 'ro', ms=8, zorder=5)
            ax.plot(-f, 0, 'ro', ms=8, zorder=5)
            ax.text(f, -0.4, "F'", fontsize=10, ha='center', color='red')
            ax.text(-f, -0.4, 'F', fontsize=10, ha='center', color='red')

            # Object (arrow at x = -do)
            ax.annotate('', xy=(-do, obj_h), xytext=(-do, 0),
                        arrowprops=dict(arrowstyle='->', color='#2ca02c', lw=2.5))
            ax.text(-do, obj_h + 0.2, 'Object', fontsize=9, ha='center', color='#2ca02c')

            # Image (if not at infinity)
            if abs(di) < 30:
                img_color = 'darkorange' if real_image else '#d62728'
                img_ls = '-' if real_image else '--'
                ax.annotate('', xy=(di, img_h), xytext=(di, 0),
                            arrowprops=dict(arrowstyle='->', color=img_color, lw=2.5,
                                            linestyle='solid' if real_image else 'dashed'))
                label = 'Real image' if real_image else 'Virtual image'
                ax.text(di, img_h + (0.3 if img_h >= 0 else -0.5), label,
                        fontsize=9, ha='center', color=img_color)

            # Principal rays
            clip_x = [x_min, x_max]

            if abs(di) < 30:
                # Ray 1: parallel to axis -> through F' (back focal point)
                ax.plot([-do, 0], [obj_h, obj_h], 'C0-', lw=1.2, alpha=0.7)
                ax.plot([0, di], [obj_h, img_h], 'C0-', lw=1.2, alpha=0.7)
                if not real_image:
                    ax.plot([0, di], [obj_h, img_h], 'C0--', lw=1, alpha=0.4)

                # Ray 2: through center — undeviated
                ax.plot([-do, di], [obj_h, img_h], 'C1-', lw=1.2, alpha=0.7)

                # Ray 3: through/toward F -> exits parallel
                if f > 0:
                    # Toward front focal point F at x = -f
                    slope3 = (obj_h - 0) / (-do - (-f)) if abs(-do + f) > 0.01 else 0
                    y_at_lens = obj_h + slope3 * (-do - (-do))
                    y_at_lens = slope3 * (0 - (-do)) + obj_h
                    # Actually: slope from object to F
                    slope3 = (0 - obj_h) / (-f - (-do))
                    y_at_lens = obj_h + slope3 * (0 - (-do))
                    ax.plot([-do, 0], [obj_h, y_at_lens], 'C2-', lw=1.2, alpha=0.7)
                    ax.plot([0, di], [y_at_lens, y_at_lens], 'C2-', lw=1.2, alpha=0.7)

            # Info text
            img_type = 'Real' if real_image else 'Virtual'
            orient = 'Inverted' if M < 0 else 'Upright'
            lens_type = 'Converging' if f > 0 else 'Diverging'
            info = (f'{lens_type} lens  |  f = {f:.1f}  |  do = {do:.1f}  |  '
                    f'di = {di:.2f}  |  M = {M:.2f}  |  {img_type}, {orient}')
            ax.set_title(info, fontsize=10)
            ax.set_xlabel('Position along optical axis')
            ax.set_ylabel('Height')

            plt.tight_layout()
            plt.show()

    for w in (do_w, f_w):
        w.observe(update, names='value')
    display(widgets.VBox([do_w, f_w, out]))
    update(None)

build_lens_demo()

## Curved Mirrors

Curved mirrors obey the **same equation** as thin lenses:

$$\frac{1}{f} = \frac{1}{d_o} + \frac{1}{d_i}, \qquad f = \frac{R}{2}$$

where $R$ is the radius of curvature of the mirror.

**Sign conventions for mirrors:**

| Mirror type | $R$ | $f$ | Image properties |
|:---|:---|:---|:---|
| **Concave** (converging) | $R > 0$ | $f > 0$ | Real & inverted if $d_o > f$; virtual & upright if $d_o < f$ |
| **Convex** (diverging) | $R < 0$ | $f < 0$ | Always virtual, upright, diminished |

**Principal rays for a concave mirror:**
1. Parallel to axis $\rightarrow$ reflects through focal point $F$
2. Through focal point $F$ $\rightarrow$ reflects parallel to axis
3. Through center of curvature $C$ $\rightarrow$ reflects back on itself

> Adjust the object position and radius of curvature. Negative $R$ gives a convex mirror.

In [ ]:
def build_mirror_demo():
    style = {'description_width': 'initial'}
    sl = dict(style=style, layout=widgets.Layout(width='420px'))
    do_w = widgets.FloatSlider(value=8, min=1, max=20, step=0.2,
                               description='Object distance do', **sl)
    R_w = widgets.FloatSlider(value=6, min=-12, max=12, step=0.2,
                              description='Radius of curvature R', **sl)
    out = widgets.Output()

    def update(_):
        do = do_w.value
        R = R_w.value
        if abs(R) < 0.1:
            R = 0.1  # avoid flat mirror edge case
        f = R / 2.0
        concave = R > 0

        if abs(1.0 / f - 1.0 / do) > 1e-6:
            di = 1.0 / (1.0 / f - 1.0 / do)
        else:
            di = 1e6
        M = -di / do
        real_image = di > 0
        obj_h = 1.5
        img_h = M * obj_h

        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(1, 1, figsize=(14, 5))

            # Mirror surface (parabolic arc)
            mirror_h = 4.0
            t = np.linspace(-mirror_h, mirror_h, 200)
            if concave:
                mirror_x = -t**2 / (2 * R)  # concave opens to the left (object side)
            else:
                mirror_x = -t**2 / (2 * R)
            ax.plot(mirror_x, t, 'gray', lw=3)
            ax.fill_betweenx(t, mirror_x, mirror_x - 0.15, color='gray', alpha=0.3)

            # Optical axis
            x_min_plot = min(-do - 3, -15)
            x_max_plot = max(3, abs(di) + 2 if abs(di) < 20 else 5)
            ax.axhline(0, color='black', lw=0.8, alpha=0.5)

            # Focal point and center of curvature
            ax.plot(-f, 0, 'ro', ms=8, zorder=5)
            ax.text(-f, -0.5, 'F', fontsize=10, ha='center', color='red')
            ax.plot(-R, 0, 'bs', ms=7, zorder=5)
            ax.text(-R, -0.5, 'C', fontsize=10, ha='center', color='blue')

            # Object
            ax.annotate('', xy=(-do, obj_h), xytext=(-do, 0),
                        arrowprops=dict(arrowstyle='->', color='#2ca02c', lw=2.5))
            ax.text(-do, obj_h + 0.3, 'Object', fontsize=9, ha='center', color='#2ca02c')

            # Image
            if abs(di) < 25:
                img_color = 'darkorange' if real_image else '#d62728'
                # For mirrors, real image is on same side as object (di > 0 means x = -di)
                img_x = -di
                ax.annotate('', xy=(img_x, img_h), xytext=(img_x, 0),
                            arrowprops=dict(arrowstyle='->', color=img_color, lw=2.5,
                                            linestyle='solid' if real_image else 'dashed'))
                label = 'Real image' if real_image else 'Virtual image'
                ax.text(img_x, img_h + (0.3 if img_h >= 0 else -0.5), label,
                        fontsize=9, ha='center', color=img_color)

                # Principal rays
                # Ray 1: parallel to axis -> reflects through F
                ax.plot([-do, 0], [obj_h, obj_h], 'C0-', lw=1.2, alpha=0.7)
                ax.plot([0, img_x], [obj_h, img_h], 'C0-', lw=1.2, alpha=0.7)

                # Ray 2: through center -> reflects back on itself
                slope_c = (obj_h) / (-do - (-R))
                y_at_mirror = obj_h + slope_c * (0 - (-do))
                ax.plot([-do, 0], [obj_h, y_at_mirror], 'C1-', lw=1.2, alpha=0.7)
                ax.plot([0, img_x], [y_at_mirror, img_h], 'C1-', lw=1.2, alpha=0.7)

                # Ray 3: through F -> exits parallel
                if concave:
                    slope_f = (obj_h) / (-do - (-f))
                    y_at_mirror_f = obj_h + slope_f * (0 - (-do))
                    ax.plot([-do, 0], [obj_h, y_at_mirror_f], 'C2-', lw=1.2, alpha=0.7)
                    ax.plot([0, img_x], [y_at_mirror_f, y_at_mirror_f], 'C2-', lw=1.2, alpha=0.7)

            # Set limits
            all_x = [-do, 0]
            if abs(di) < 25:
                all_x.append(-di)
            ax.set_xlim(min(all_x) - 3, max(max(all_x) + 3, 3))
            y_lim = max(abs(obj_h), abs(img_h) if abs(img_h) < 10 else obj_h, mirror_h) + 1
            ax.set_ylim(-y_lim, y_lim)

            # Info
            mirror_type = 'Concave' if concave else 'Convex'
            img_type = 'Real' if real_image else 'Virtual'
            orient = 'Inverted' if M < 0 else 'Upright'
            info = (f'{mirror_type} mirror  |  R = {R:.1f}  |  f = {f:.1f}  |  '
                    f'do = {do:.1f}  |  di = {di:.2f}  |  M = {M:.2f}  |  {img_type}, {orient}')
            ax.set_title(info, fontsize=10)
            ax.set_xlabel('Position along optical axis')
            ax.set_ylabel('Height')

            plt.tight_layout()
            plt.show()

    for w in (do_w, R_w):
        w.observe(update, names='value')
    display(widgets.VBox([do_w, R_w, out]))
    update(None)

build_mirror_demo()

## Interference — Young's Double Slit

When coherent light passes through two narrow slits separated by distance $d$, the waves from each slit interfere on a distant screen. The **path difference** between the two waves at angle $\theta$ is:

$$\Delta = d \sin\theta$$

**Constructive interference** (bright fringes) occurs when:

$$d \sin\theta = m\lambda, \qquad m = 0, \pm 1, \pm 2, \ldots$$

**Destructive interference** (dark fringes) occurs when:

$$d \sin\theta = \left(m + \tfrac{1}{2}\right)\lambda$$

For small angles ($\sin\theta \approx \tan\theta = y/L$), the fringe spacing on a screen at distance $L$ is:

$$\Delta y = \frac{\lambda L}{d}$$

The intensity pattern from two identical slits is:

$$I(\theta) = 4 I_0 \cos^2\!\left(\frac{\pi d \sin\theta}{\lambda}\right)$$

> Adjust the slit separation $d$, wavelength $\lambda$, and screen distance $L$. The left panel shows the 2D wave pattern; the right panel shows the intensity on the screen.

In [ ]:
def build_double_slit_demo():
    style = {'description_width': 'initial'}
    sl = dict(style=style, layout=widgets.Layout(width='420px'))
    d_w = widgets.FloatSlider(value=5.0, min=1.0, max=20.0, step=0.5,
                              description='Slit separation d (um)', **sl)
    lam_w = widgets.FloatSlider(value=550, min=380, max=750, step=10,
                                description='Wavelength (nm)', **sl)
    L_w = widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1,
                              description='Screen distance L (m)', **sl)
    out = widgets.Output()

    def update(_):
        d_um = d_w.value
        lam_nm = lam_w.value
        L = L_w.value
        d = d_um * 1e-6   # meters
        lam = lam_nm * 1e-9  # meters
        fringe_spacing = lam * L / d * 1e3  # in mm

        with out:
            clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

            # Left panel: 2D wave interference pattern
            extent = 40 * lam * 1e6  # in micrometers scale
            x = np.linspace(0, extent, 300)
            y = np.linspace(-extent / 2, extent / 2, 300)
            X, Y = np.meshgrid(x, y)

            # Two sources at (0, +d/2) and (0, -d/2) in um
            d_plot = d * 1e6  # in um
            r1 = np.sqrt(X**2 + (Y - d_plot / 2)**2)
            r2 = np.sqrt(X**2 + (Y + d_plot / 2)**2)
            k = 2 * np.pi / (lam * 1e6)  # k in 1/um
            wave = np.cos(k * r1) + np.cos(k * r2)

            ax1.imshow(wave, extent=[0, extent, -extent / 2, extent / 2],
                       cmap='RdBu', aspect='auto', origin='lower')
            ax1.plot(0, d_plot / 2, 'ko', ms=5, zorder=5)
            ax1.plot(0, -d_plot / 2, 'ko', ms=5, zorder=5)
            ax1.set_xlabel('x (um)')
            ax1.set_ylabel('y (um)')
            ax1.set_title(f'2D Wave Pattern from Two Sources (d = {d_um:.1f} um)')

            # Right panel: intensity on screen
            y_screen = np.linspace(-15, 15, 1000)  # in mm
            theta = np.arctan(y_screen * 1e-3 / L)
            phase = np.pi * d * np.sin(theta) / lam
            I = 4 * np.cos(phase)**2

            ax2.plot(I, y_screen, 'steelblue', lw=2)
            ax2.fill_betweenx(y_screen, I, alpha=0.2, color='steelblue')
            ax2.set_xlabel('Intensity (normalized)')
            ax2.set_ylabel('Position on screen y (mm)')
            ax2.set_title(f'Fringe pattern  |  spacing = {fringe_spacing:.2f} mm')
            ax2.set_xlim(0, 4.5)

            # Mark bright fringes
            for m in range(-5, 6):
                y_m = m * lam * L / d * 1e3  # mm
                if abs(y_m) < 15:
                    ax2.axhline(y_m, color='red', ls=':', lw=0.8, alpha=0.5)
                    if m == 0:
                        ax2.text(4.2, y_m, f'm={m}', fontsize=8, color='red', va='center')
                    elif abs(m) <= 3:
                        ax2.text(4.2, y_m, f'{m}', fontsize=8, color='red', va='center')

            plt.tight_layout()
            plt.show()

    for w in (d_w, lam_w, L_w):
        w.observe(update, names='value')
    display(widgets.VBox([d_w, lam_w, L_w, out]))
    update(None)

build_double_slit_demo()